In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚨 Incident Response & On-Call Management Guide

**Handling production incidents and building reliable incident response processes**

This guide covers:
- Incident classification and severity levels
- On-call rotations and scheduling
- Alert fatigue reduction
- Post-incident reviews (blameless)
- Communication during incidents
- Documentation and runbooks
- Metrics and SLOs
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Incident Classification

### Severity Levels
```
P0 - CRITICAL (Severity: Critical)
  • Complete service outage
  • Data loss or corruption
  • Security breach active
  • Response time: <15 minutes
  • Escalation: Yes
  • Example: Authentication service down
  
  Action:
  - Page on-call immediately
  - Start war room
  - Begin communication cadence

P1 - HIGH (Severity: High)
  • Partial service degradation
  • Some users affected
  • Database connection pool exhausted
  • Response time: <30 minutes
  • Escalation: Yes
  
  Action:
  - Page on-call
  - Investigate while monitoring
  - Update status page

P2 - MEDIUM (Severity: Medium)
  • Minor feature broken
  • Performance degradation
  • One component affected
  • Response time: <2 hours
  • Escalation: No
  
  Action:
  - Notify team
  - Schedule fix
  - Document issue

P3 - LOW (Severity: Low)
  • Non-critical feature
  • Cosmetic issues
  • Single user impact
  • Response time: Normal hours
  
  Action:
  - Log ticket
  - Add to backlog
```

### Impact Assessment
```python
def assess_incident_severity(metrics):
    """Auto-calculate severity"""
    
    affected_users_pct = metrics['affected_users'] / metrics['total_users']
    error_rate = metrics['error_rate']
    duration = metrics['duration_minutes']
    
    if affected_users_pct > 0.5 or error_rate > 50:
        return "P0"
    elif affected_users_pct > 0.1 or error_rate > 10:
        return "P1"
    elif affected_users_pct > 0.01 or error_rate > 1:
        return "P2"
    else:
        return "P3"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. On-Call Rotation Management

### Scheduling Strategy
```python
import datetime
from typing import List, Dict

class OnCallSchedule:
    def __init__(self, engineers: List[str], rotation_days: int = 7):
        self.engineers = engineers
        self.rotation_days = rotation_days
    
    def get_oncall_engineer(self, date: datetime.date = None) -> str:
        """Get on-call engineer for date"""
        if date is None:
            date = datetime.date.today()
        
        # Rotate every 7 days
        days_since_start = (date - datetime.date(2024, 1, 1)).days
        index = (days_since_start // self.rotation_days) % len(self.engineers)
        
        return self.engineers[index]
    
    def get_backup_engineer(self, date: datetime.date = None) -> str:
        """Get backup for coverage"""
        primary = self.get_oncall_engineer(date)
        next_index = (self.engineers.index(primary) + 1) % len(self.engineers)
        return self.engineers[next_index]

# Usage
schedule = OnCallSchedule(['alice', 'bob', 'charlie', 'diana'])
print(f"Today: {schedule.get_oncall_engineer()}")
print(f"Backup: {schedule.get_backup_engineer()}")
```

### Preventing Burnout
```python
# Rules
ONCALL_RULES = {
    'max_consecutive_weeks': 2,
    'min_rest_weeks_between': 2,
    'max_oncalls_per_quarter': 6,
    'no_weekend_more_than_once_per_month': True,
    'compensation_hours_per_incident': {
        'P0': 4,
        'P1': 2,
        'P2': 0.5
    }
}

# Track burden
def calculate_oncall_burden(engineer, incidents_this_month):
    p0_incidents = sum(1 for i in incidents_this_month if i['severity'] == 'P0')
    p1_incidents = sum(1 for i in incidents_this_month if i['severity'] == 'P1')
    
    burden = (p0_incidents * 4) + (p1_incidents * 2)
    
    if burden > 10:
        print(f"⚠️  {engineer} is overburden ({burden} hours)")
        return "REASSIGN_NEXT_WEEK"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Alert Management & Alert Fatigue

### Alert Rules (PagerDuty Integration)
```yaml
# Alert routing rules
alert_rules:
  database_connection_pool:
    threshold: 80%
    duration: 5m
    severity: P1
    pages: ["database-team"]
    fallback: ["on-call-primary"]
  
  high_error_rate:
    threshold: 5% error rate
    duration: 2m
    severity: P2
    pages: ["backend-team"]
  
  cpu_usage:
    threshold: 85%
    duration: 10m
    severity: P2
    action: "auto-scale"
```

### Alert Consolidation (Reduce Noise)
```python
class AlertDeduplicator:
    def __init__(self, lookback_minutes: int = 30):
        self.lookback_minutes = lookback_minutes
        self.recent_alerts = {}
    
    def should_page(self, alert):
        """Deduplicate similar alerts"""
        alert_key = f"{alert['service']}:{alert['check']}"
        
        # Check if similar alert was sent recently
        if alert_key in self.recent_alerts:
            last_alert_time = self.recent_alerts[alert_key]
            if (datetime.now() - last_alert_time).seconds < self.lookback_minutes * 60:
                return False  # Skip duplicate
        
        # Record this alert
        self.recent_alerts[alert_key] = datetime.now()
        return True

# Smart grouping
def group_alerts(alerts):
    """Group related alerts for context"""
    
    groups = {}
    for alert in alerts:
        service = alert['service']
        
        if service not in groups:
            groups[service] = []
        
        groups[service].append(alert)
    
    # Only page if 3+ alerts from same service (indicates real issue)
    return {
        service: alerts
        for service, alerts in groups.items()
        if len(alerts) >= 3
    }
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 4. Incident Communication

### Escalation Process
```
Incident Detected
    ↓
Severity Assessment (P0/P1/P2/P3)
    ↓
Page on-call → Acknowledge (5 min SLA)
    ↓
[If not acknowledged in 5 min]
    ↓
Page backup → Acknowledge (5 min SLA)
    ↓
[If not acknowledged in 10 min]
    ↓
Page manager
    ↓
Start incident command (war room)
```

### War Room Protocol
```python
import time
from datetime import datetime

class IncidentWarRoom:
    def __init__(self, incident_id: str):
        self.incident_id = incident_id
        self.started_at = datetime.now()
        self.communications = []
    
    def start_war_room(self):
        """Initiate incident response"""
        print("🚨 INCIDENT WAR ROOM STARTED")
        print(f"Incident ID: {self.incident_id}")
        print(f"Start time: {self.started_at}")
        print("---")
        print("Roles:")
        print("  Incident Commander: alice")
        print("  Engineering Lead: bob")
        print("  Comms Lead: charlie")
    
    def broadcast_update(self, update: str, author: str):
        """Broadcast update to all stakeholders"""
        self.communications.append({
            'time': datetime.now(),
            'author': author,
            'message': update
        })
        
        print(f"[{author}] {update}")
        
        # Send to Slack, email, status page
        self.notify_channels(update)
    
    def get_duration(self) -> str:
        """Time since incident started"""
        delta = datetime.now() - self.started_at
        minutes = delta.total_seconds() / 60
        return f"{int(minutes)}m {int(delta.seconds % 60)}s"
```

### Communication Templates
```
🚨 INCIDENT NOTIFICATION
Incident ID: INC-2024-001
Severity: P1
Service: Payment API
Status: Investigating

Affected: ~5% of users
Started: 2024-01-15 14:32 UTC
Duration: 15 minutes

Impact: Payments failing intermittently
Workaround: None available
Next update: In 10 minutes

IRC: #incidents
```
</MySCode.Cell>

<MySCode.Cell language="markdown">
## 5. Post-Incident Review (Blameless)

```python
class PostIncidentReview:
    def __init__(self, incident_id: str):
        self.incident_id = incident_id
        self.review = {
            'timeline': [],
            'root_causes': [],
            'lessons_learned': [],
            'action_items': []
        }
    
    def build_timeline(self):
        """Reconstruct incident timeline from logs"""
        return [
            ('14:32:00', 'Payment API error rate increased'),
            ('14:32:15', 'Automated alert triggered'),
            ('14:33:00', 'On-call engineer acknowledged'),
            ('14:35:00', 'Identified database connection pool exhaustion'),
            ('14:37:00', 'Restarted database service'),
            ('14:38:00', 'Error rate returned to normal'),
            ('14:40:00', 'Incident resolved')
        ]
    
    def identify_root_causes(self):
        """Blameless root cause analysis"""
        
        # Focus on systems, not people
        causes = [
            "Connection pool max size was set to 50 (low)",
            "No alert for connection pool saturation",
            "Database query optimization was pending for 3 weeks",
            "Load testing didn't simulate peak traffic"
        ]
        
        # What could have prevented this?
        preventative = [
            "Increase connection pool to 200",
            "Add monitoring for pool saturation (>80%)",
            "Prioritize query optimization",
            "Implement realistic load testing"
        ]
        
        return causes, preventative
    
    def generate_report(self):
        """Create blameless post-mortem"""
        
        report = f"""
# Post-Incident Review: {self.incident_id}

## Summary
Payment API was unavailable for 8 minutes due to database connection pool exhaustion.

## Timeline
{self._format_timeline()}

## Root Causes
{self._format_causes()}

## Action Items
- [ ] Increase database connection pool to 200
- [ ] Add monitoring alert for >80% pool utilization
- [ ] Complete database query optimization (Priority)
- [ ] Implement load testing for 2x peak traffic
- [ ] Review on-call response procedures

## Lessons Learned
1. Connection pool sizing was insufficient
2. Monitoring gaps around resource exhaustion
3. Load testing didn't reflect real-world traffic patterns

## Prevention
- [ ] Quarterly review of resource limits
- [ ] Monthly load testing
- [ ] Improved alerting rules
"""
        return report
```
</MySCode.Cell>
```